# Feature Engineering

### Imports

In [1]:
import pandas as pd
import geopandas as gpd
import zipfile
import sys
from pathlib import Path
sys.path.append(str(Path("..").resolve()))
from src.config import RAIN_COLS_HEB_TO_ENG
from src.feature_engineering import modify_rainfall_df_values, group_rainfall_by_day_and_hour, merge_rain_to_trips_df
from src.feature_engineering import route_to_shape_dict, get_linestring_for_shape, add_linestrings_to_route_dict,extract_linestring
from src.feature_engineering import buffer_and_dissolve_routes, calc_length_within_buffer, calc_curvity, parse_mixed_date, filter_by_multiple_date_windows

In [3]:
### Import Trips interim dataframe
df_trips = pd.read_csv(r'../data/interim_data/telaviv_buses_0501_1801_2024_cleaned.csv')

### Rainfall
Rainfall data was obtained from the Israeli Meteorological Service (IMS) open data portal:
https://ims.gov.il/he/data_gov

In [4]:
### Importing rainfall data collected by the Israeli Meteorological Service (IMS)
df_rain = pd.read_csv(r'../data/external/rainfall_tlv_jan_2024.csv')

### Renaming its columns to english for easy processing
df_rain = df_rain.rename(columns=RAIN_COLS_HEB_TO_ENG)

### Modifying value types
df_rain = modify_rainfall_df_values(df_rain)

### Grouping (sum) rainfall by day and hour 
rain_grouped_df = group_rainfall_by_day_and_hour(df_rain)

### Merge the rainfall (mm) to th trips df - so each trip will have the amount of rainfall at the time (hour) of the trip
df_trips = merge_rain_to_trips_df(rain_grouped_df,df_trips)

## Spatial attributes

### Imports

In [5]:
#### public transport paths
public_transport_paths = gpd.read_file("../data/external/shapefiles/Nataz_shape_202601.zip")


#### gtfs data
zip_path = '../data/external/gtfs/gtfs.zip'
with zipfile.ZipFile(zip_path) as z:
    print(z.namelist())  # see file names inside

    df_trips_gtfs = pd.read_csv(z.open('trips.txt'))
    df_shapes_gtfs = pd.read_csv(z.open('shapes.txt'))
    df_routes_gtfs = pd.read_csv(z.open('routes.txt'))

['routes.txt', 'shapes.txt', 'trips.txt']


### Process

- Assign to each trips its geometry
- Convert  the trips df to GDF (GeoDataFrame) based on the route geometry (linestring)

In [6]:
### Create a dictionary to each route_id the associated shape_id in gtfs {'route_id_1': {'shape_id': shape_id_1},...}                                                                                    
route_shape_map  = route_to_shape_dict(df_trips_gtfs, df_trips.route_id.unique())

### Add to the route_shape_map to each route_id the linestinrg extracted from shape_id df {'shape_id': shape_id_1, 'linestring': 'linestring_1},...} 
route_shape_line_map = add_linestrings_to_route_dict(df_shapes_gtfs, route_shape_map)

### Add this linestring to the trips df as an attribute - "linestring"
df_trips["linestring"] = df_trips["route_id"].map(lambda x: extract_linestring(x, route_shape_line_map))

### Convert the df to GeoDataFrame, using the linestring as the geometry reference
gdf_trips = gpd.GeoDataFrame(df_trips, geometry="linestring", crs="EPSG:4326")

### Handle CRS - make it in ITM for better distance measuring
gdf_trips = gdf_trips.set_crs(epsg=4326)   # only if not already set
gdf_trips = gdf_trips.to_crs(epsg=2039)

- Handle The Public Transport Paths data
- Filter paths that were removed before, or added after the investigated timeframe

In [7]:
public_transport_routes = public_transport_paths.set_crs(epsg=2039, allow_override=True)

cols = ["STARTDATE", "ENDDATE"]
public_transport_paths[cols] = public_transport_paths[cols].apply(lambda col: col.apply(parse_mixed_date))

public_transport_routes_filtered= filter_by_multiple_date_windows(
                                                                public_transport_paths,
                                                                start_cols=["STARTDATE"],
                                                                end_cols=["ENDDATE"],
                                                                start_cutoffs=["2024-01-01"],
                                                                end_cutoffs=["2024-02-01"]
)

- Adding spatial attributes
- Calculating the length within public transport path (based on buffer intersection with it)
- Adding the percentage of the route within this path
- Adding curvity metric (route_length/aerial distance between start and end stops)

In [8]:
### Making 15 m buffer around each public transport path
pt_routes_area_gdf = buffer_and_dissolve_routes(public_transport_routes_filtered, buffer_m=15)

### Dissolving all the geometry into one polygon
buffer_geom = pt_routes_area_gdf.geometry.iloc[0]

### Intersection each route linestring, and calculating the sum og the linestring inside the buffer
### lines lower than 200m removed
gdf_trips["length_in_buffer_m"] = gdf_trips["linestring"].apply(
    lambda geom: calc_length_within_buffer(geom, buffer_geom, min_length=200)
)

### Adding the gtfs route_length attribut in meters
gdf_trips['route_length'] = gdf_trips.length

### Calculating the percetnage of the distance inside the public transport path from the entire route length
gdf_trips['perc_within_pt_route'] = round(gdf_trips['length_in_buffer_m']/gdf_trips['route_length']*100,2)

### Curvity metric
gdf_trips["curvity"] = gdf_trips["linestring"].apply(calc_curvity)

## Other attributes

- Adding one attribute that will consist -agency + line number + direction-
- Urban - boolean  (more than 25km)

In [9]:
gdf_trips['line_num_agency_alter_dir'] = gdf_trips['agency_name'] +'_'+ gdf_trips['line_num'] +'_'+ gdf_trips['direction'].astype(str)
gdf_trips['urban'] = gdf_trips['route_length'] <=25

In [12]:
gdf_trips.drop(columns = ["linestring"]).to_csv(r'../data/interim_data/telaviv_buses_0501_1801_2024_cleaned_new_features.csv',index=False,encoding='utf-8-sig')

In [19]:

gdf_trips[['route_id','linestring']].drop_duplicates(subset=['route_id','linestring']).to_csv(r'../data/interim_data/routes_linestrings.csv',index=False,encoding='utf-8-sig')

In [3]:
df = pd.read_csv(r'../data/interim_data/telaviv_buses_0501_1801_2024_cleaned_new_features.csv',encoding='utf-8-sig')
import json

path = r'C:\Users\shaha\Documents\DS_course\public_transport_ML\data\others\route_id_to_mkt_id.txt'  # or .json if you changed it

with open(path, 'r') as f:
    d = json.load(f)

d_int = {int(k): v for k, v in d.items()}
df['route_mkt'] = df['route_id'].map(d_int)

cols = df.columns.tolist()
cols.insert(6, cols.pop(cols.index('route_mkt')))
df = df[cols]
df['route_mkt'] = df['route_mkt'].astype('Int64')
df.to_csv(r'../data/interim_data/telaviv_buses_0501_1801_2024_cleaned_new_features.csv',index=False,encoding='utf-8-sig')

In [2]:
df = pd.read_csv(r'../data/interim_data/telaviv_buses_0501_1801_2024_cleaned_new_features.csv',encoding='utf-8-sig')


In [3]:
df.head()

,date,day,hour_rounded,line_num,line_name,route_id,route_mkt,direction,alternative,agency_name,...,speed_kmh_planned,speed_kmh_actual,SIRI_id,rainfall_mm,length_in_buffer_m,route_length,perc_within_pt_route,curvity,line_num_agency_alter_dir,urban
0,2024-01-05,Friday,0,61,הכובשים/דניאל-תל אביב יפו<->מסוף עמידר-רמת גן-10,2510,16061.0,1.0,0,Dan,...,18.0,15.3,58601679,0.0,1528.828626,12797.049700,11.95,1.843115,Dan_61_1.0,False
1,2024-01-05,Friday,0,2,פרופ' ישעיהו ליבוביץ'/הרב יעקב טראב-תל אביב יפ...,27082,99002.0,2.0,0,Dan,...,18.0,19.7,58601680,0.0,5074.525101,19588.509346,25.91,2.517371,Dan_2_2.0,False
2,2024-01-05,Friday,0,699,הרכבת/ראש פינה-תל אביב יפו<->ת. מרכזית נתניה-נ...,22058,11699.0,2.0,#,Metropolin,...,29.9,29.9,58601681,0.0,21555.169188,35080.611916,61.44,1.149308,Metropolin_699_2.0,False
3,2024-01-05,Friday,0,555,מסוף רדינג/רציפים-תל אביב יפו<->מסוף יהוד/הורד...,1048,11555.0,1.0,0,Kavim,...,19.8,22.2,58601684,0.0,7372.369073,20643.347935,35.71,1.561211,Kavim_555_1.0,False
4,2024-01-05,Friday,0,8,מסוף אונברסיטת ת''א-תל אביב יפו<->מסוף הטייסים...,16076,55008.0,1.0,0,Dan,...,16.3,17.7,58601687,0.0,5123.405002,14665.598732,34.93,1.916634,Dan_8_1.0,False


In [4]:
val = pd.read_csv(r'../data/external/grouped_validations.csv',encoding='utf-8-sig')
val.head()

C:\Users\Tamir\AppData\Local\Temp\ipykernel_26128\118090289.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  val = pd.read_csv(r'../data/external/grouped_validations.csv',encoding='utf-8-sig')


,Alternative,RouteId,Direction,FullHour,DayName,Total_Passengers,Bus_Count,Avg_Passengers_Per_Bus
0,#,10001,3,3.0,Friday,42,2,21.000000
1,#,10001,3,3.0,Monday,18,3,6.000000
2,#,10001,3,3.0,Sunday,41,3,13.666667
3,#,10001,3,3.0,Thursday,31,3,10.333333
4,#,10001,3,3.0,Tuesday,47,3,15.666667


In [15]:
val['route_dir_alt_day_hr'] = (
    val['RouteId'].astype(int).astype(str) + '_' +
    val['Direction'].astype(int).astype(str) + '_' +
    val['Alternative'].astype(str) + '_' +
    val['DayName'].astype(str) + '_' +
    val['FullHour'].astype(int).astype(str)
)

val.head()

,Alternative,RouteId,Direction,FullHour,DayName,Total_Passengers,Bus_Count,Avg_Passengers_Per_Bus,route_dir_alt_day_hr
0,#,10001,3,3.0,Friday,42,2,21.000000,10001_3_#_Friday_3
1,#,10001,3,3.0,Monday,18,3,6.000000,10001_3_#_Monday_3
2,#,10001,3,3.0,Sunday,41,3,13.666667,10001_3_#_Sunday_3
3,#,10001,3,3.0,Thursday,31,3,10.333333,10001_3_#_Thursday_3
4,#,10001,3,3.0,Tuesday,47,3,15.666667,10001_3_#_Tuesday_3


In [19]:
df['route_dir_alt_day_hr'] = (
    df['route_mkt'].fillna(0).astype(int).astype(str) + '_' +
    df['direction'].fillna(0).astype(int).astype(str) + '_' +
    df['alternative'].astype(str) + '_' +
    df['day'].astype(str) + '_' +
    df['hour_rounded'].fillna(0).astype(int).astype(str)
)

df.head()

,date,day,hour_rounded,line_num,line_name,route_id,route_mkt,direction,alternative,agency_name,...,speed_kmh_actual,SIRI_id,rainfall_mm,length_in_buffer_m,route_length,perc_within_pt_route,curvity,line_num_agency_alter_dir,urban,route_dir_alt_day_hr
0,2024-01-05,Friday,0,61,הכובשים/דניאל-תל אביב יפו<->מסוף עמידר-רמת גן-10,2510,16061.0,1.0,0,Dan,...,15.3,58601679,0.0,1528.828626,12797.049700,11.95,1.843115,Dan_61_1.0,False,16061_1_0_Friday_0
1,2024-01-05,Friday,0,2,פרופ' ישעיהו ליבוביץ'/הרב יעקב טראב-תל אביב יפ...,27082,99002.0,2.0,0,Dan,...,19.7,58601680,0.0,5074.525101,19588.509346,25.91,2.517371,Dan_2_2.0,False,99002_2_0_Friday_0
2,2024-01-05,Friday,0,699,הרכבת/ראש פינה-תל אביב יפו<->ת. מרכזית נתניה-נ...,22058,11699.0,2.0,#,Metropolin,...,29.9,58601681,0.0,21555.169188,35080.611916,61.44,1.149308,Metropolin_699_2.0,False,11699_2_#_Friday_0
3,2024-01-05,Friday,0,555,מסוף רדינג/רציפים-תל אביב יפו<->מסוף יהוד/הורד...,1048,11555.0,1.0,0,Kavim,...,22.2,58601684,0.0,7372.369073,20643.347935,35.71,1.561211,Kavim_555_1.0,False,11555_1_0_Friday_0
4,2024-01-05,Friday,0,8,מסוף אונברסיטת ת''א-תל אביב יפו<->מסוף הטייסים...,16076,55008.0,1.0,0,Dan,...,17.7,58601687,0.0,5123.405002,14665.598732,34.93,1.916634,Dan_8_1.0,False,55008_1_0_Friday_0


In [24]:
df = pd.merge(
    df,
    val[['route_dir_alt_day_hr', 'Total_Passengers', 'Avg_Passengers_Per_Bus']],
    on='route_dir_alt_day_hr',
    how='left'
)

print(f"גודל הטבלה אחרי merge: {df.shape}")
print(df[['route_dir_alt_day_hr', 'Total_Passengers', 'Avg_Passengers_Per_Bus']].head())

MergeError: Passing 'suffixes' which cause duplicate columns {'Total_Passengers_x'} is not allowed.

In [26]:
df.head()

,date,day,hour_rounded,line_num,line_name,route_id,route_mkt,direction,alternative,agency_name,...,route_length,perc_within_pt_route,curvity,line_num_agency_alter_dir,urban,route_dir_alt_day_hr,Total_Passengers_x,Total_Passengers_y,Total_Passengers,Avg_Passengers_Per_Bus
0,2024-01-05,Friday,0,61,הכובשים/דניאל-תל אביב יפו<->מסוף עמידר-רמת גן-10,2510,16061.0,1.0,0,Dan,...,12797.049700,11.95,1.843115,Dan_61_1.0,False,16061_1_0_Friday_0,87.0,87.0,87.0,29.0
1,2024-01-05,Friday,0,2,פרופ' ישעיהו ליבוביץ'/הרב יעקב טראב-תל אביב יפ...,27082,99002.0,2.0,0,Dan,...,19588.509346,25.91,2.517371,Dan_2_2.0,False,99002_2_0_Friday_0,11.0,11.0,11.0,5.5
2,2024-01-05,Friday,0,699,הרכבת/ראש פינה-תל אביב יפו<->ת. מרכזית נתניה-נ...,22058,11699.0,2.0,#,Metropolin,...,35080.611916,61.44,1.149308,Metropolin_699_2.0,False,11699_2_#_Friday_0,1.0,1.0,1.0,1.0
3,2024-01-05,Friday,0,555,מסוף רדינג/רציפים-תל אביב יפו<->מסוף יהוד/הורד...,1048,11555.0,1.0,0,Kavim,...,20643.347935,35.71,1.561211,Kavim_555_1.0,False,11555_1_0_Friday_0,NaN,NaN,NaN,NaN
4,2024-01-05,Friday,0,8,מסוף אונברסיטת ת''א-תל אביב יפו<->מסוף הטייסים...,16076,55008.0,1.0,0,Dan,...,14665.598732,34.93,1.916634,Dan_8_1.0,False,55008_1_0_Friday_0,17.0,17.0,17.0,8.5


In [27]:
df = df.drop(columns=['Total_Passengers_x', 'Total_Passengers_y'], errors='ignore')

print(df.shape)
print(df.columns.tolist())

(214634, 35)
['date', 'day', 'hour_rounded', 'line_num', 'line_name', 'route_id', 'route_mkt', 'direction', 'alternative', 'agency_name', 'route_type', 'origin_city', 'origin_station', 'destination_city', 'destination_station', 'number_of_stops', 'route_length_km', 'departure_time_planned', 'arrival_time_planned', 'duration_min_planned', 'duration_min_actual', 'duration_difference_min', 'speed_kmh_planned', 'speed_kmh_actual', 'SIRI_id', 'rainfall_mm', 'length_in_buffer_m', 'route_length', 'perc_within_pt_route', 'curvity', 'line_num_agency_alter_dir', 'urban', 'route_dir_alt_day_hr', 'Total_Passengers', 'Avg_Passengers_Per_Bus']


In [28]:
df.head()

,date,day,hour_rounded,line_num,line_name,route_id,route_mkt,direction,alternative,agency_name,...,rainfall_mm,length_in_buffer_m,route_length,perc_within_pt_route,curvity,line_num_agency_alter_dir,urban,route_dir_alt_day_hr,Total_Passengers,Avg_Passengers_Per_Bus
0,2024-01-05,Friday,0,61,הכובשים/דניאל-תל אביב יפו<->מסוף עמידר-רמת גן-10,2510,16061.0,1.0,0,Dan,...,0.0,1528.828626,12797.049700,11.95,1.843115,Dan_61_1.0,False,16061_1_0_Friday_0,87.0,29.0
1,2024-01-05,Friday,0,2,פרופ' ישעיהו ליבוביץ'/הרב יעקב טראב-תל אביב יפ...,27082,99002.0,2.0,0,Dan,...,0.0,5074.525101,19588.509346,25.91,2.517371,Dan_2_2.0,False,99002_2_0_Friday_0,11.0,5.5
2,2024-01-05,Friday,0,699,הרכבת/ראש פינה-תל אביב יפו<->ת. מרכזית נתניה-נ...,22058,11699.0,2.0,#,Metropolin,...,0.0,21555.169188,35080.611916,61.44,1.149308,Metropolin_699_2.0,False,11699_2_#_Friday_0,1.0,1.0
3,2024-01-05,Friday,0,555,מסוף רדינג/רציפים-תל אביב יפו<->מסוף יהוד/הורד...,1048,11555.0,1.0,0,Kavim,...,0.0,7372.369073,20643.347935,35.71,1.561211,Kavim_555_1.0,False,11555_1_0_Friday_0,NaN,NaN
4,2024-01-05,Friday,0,8,מסוף אונברסיטת ת''א-תל אביב יפו<->מסוף הטייסים...,16076,55008.0,1.0,0,Dan,...,0.0,5123.405002,14665.598732,34.93,1.916634,Dan_8_1.0,False,55008_1_0_Friday_0,17.0,8.5


In [30]:
df.to_csv(r'../data/interim_data/telaviv_buses_0501_1801_2024_cleaned_new_features.csv',index=False,encoding='utf-8-sig')

In [31]:
import os
print(os.path.exists(r'C:\Users\Tamir\OneDrive - אליה הנדסה בע מ 100514861\Desktop\Tamir\pythonProj\public_transport_ML\data\interim_data\telaviv_buses_0501_1801_2024_cleaned_new_features.csv'))

True
